In [1]:
!pip install pyaudio

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install soundcard numpy


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/43.9 kB ? eta -:--:--
   --------------------------- ------------ 30.7/43.9 kB 660.6 kB/s eta 0:00:01
   ---------------------------------------- 43.9/43.9 kB 717.0 kB/s eta 0:00:00


In [6]:
import soundcard as sc
import numpy as np
import socket
import sys
import threading
# ==========================================
# CONFIGURAÇÕES DE REDE
# ==========================================
ESP32_IP = '192.168.0.176'  # O IP atualizado do seu ESP32
PORT = 7000
CHUNK = 512  # Tamanho do pacote (baixa latência)

In [7]:
def discard_incoming_data(sock):
    """
    O ESP32 envia dados de áudio (Line-in) o tempo todo. 
    Se não lermos, o buffer TCP do ESP32 enche e ele trava/reinicia (Erro 10054).
    Esta função funciona como um 'ralo' que lê e descarta esses dados silenciosamente.
    """
    try:
        while True:
            # Lê blocos de 4KB por vez e joga no vazio
            data = sock.recv(4096)
            if not data:
                break
    except Exception:
        pass

In [8]:
def main():
    print(f"Tentando conectar ao hub de áudio em {ESP32_IP}:{PORT}...")
    
    # 1. Inicia a conexão TCP
    try:
        client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        client_socket.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1) 
        client_socket.connect((ESP32_IP, PORT))
        print("[SUCESSO] Conectado ao ESP32!")
    except Exception as e:
        print(f"[ERRO] Não foi possível conectar: {e}")
        sys.exit(1)

    # Inicia a Thread que limpa o buffer TCP
    t_recv = threading.Thread(target=discard_incoming_data, args=(client_socket,))
    t_recv.daemon = True
    t_recv.start()

    # 2. Configura a captura do áudio do sistema (Loopback)
    try:
        speaker = sc.default_speaker()
        loopback_mic = sc.get_microphone(id=str(speaker.name), include_loopback=True)
        print(f"[ÁUDIO] Capturando a saída do sistema: {speaker.name}")
    except Exception as e:
        print(f"[ERRO] Falha ao configurar o Loopback de áudio: {e}")
        client_socket.close()
        sys.exit(1)

    print("\n>>> TRANSMISSÃO NATIVA DE SISTEMA (SPOTIFY, YOUTUBE) ATIVA <<<")
    print("Coloque uma música para tocar! Pressione Ctrl+C para encerrar.\n")

    try:
        # 3. Inicia a gravação e envio
        with loopback_mic.recorder(samplerate=44100, channels=2) as mic:
            while True:
                data = mic.record(numframes=CHUNK)
                # Converte float32 (-1.0 a 1.0) para int16
                pcm_data = (data * 32767.0).astype(np.int16)
                
                # Envia para o ESP32
                client_socket.sendall(pcm_data.tobytes())

    except KeyboardInterrupt:
        print("\n[ENCERRANDO] Finalizando a transmissão de forma segura...")
    except Exception as e:
        print(f"\n[ERRO DURANTE STREAMING] A conexão caiu ou o áudio falhou: {e}")
    finally:
        client_socket.close()
        print("Conexão fechada.")

if __name__ == "__main__":
    main()

Tentando conectar ao hub de áudio em 192.168.0.176:7000...
[SUCESSO] Conectado ao ESP32!
[ÁUDIO] Capturando a saída do sistema: Altofalantes (Realtek(R) Audio)

>>> TRANSMISSÃO NATIVA DE SISTEMA (SPOTIFY, YOUTUBE) ATIVA <<<
Coloque uma música para tocar! Pressione Ctrl+C para encerrar.



C:\Users\levyg\AppData\Roaming\Python\Python312\site-packages\soundcard\mediafoundation.py:772: SoundcardRuntimeWarning: data discontinuity in recording
  warnings.warn("data discontinuity in recording", SoundcardRuntimeWarning)
C:\Users\levyg\AppData\Roaming\Python\Python312\site-packages\soundcard\mediafoundation.py:772: SoundcardRuntimeWarning: data discontinuity in recording
  warnings.warn("data discontinuity in recording", SoundcardRuntimeWarning)
C:\Users\levyg\AppData\Roaming\Python\Python312\site-packages\soundcard\mediafoundation.py:772: SoundcardRuntimeWarning: data discontinuity in recording
  warnings.warn("data discontinuity in recording", SoundcardRuntimeWarning)
C:\Users\levyg\AppData\Roaming\Python\Python312\site-packages\soundcard\mediafoundation.py:772: SoundcardRuntimeWarning: data discontinuity in recording
  warnings.warn("data discontinuity in recording", SoundcardRuntimeWarning)
C:\Users\levyg\AppData\Roaming\Python\Python312\site-packages\soundcard\mediafoundat


[ENCERRANDO] Finalizando a transmissão de forma segura...
Conexão fechada.
